# Business Entity Resolution: 2-minute demo

This notebook runs the **full pipeline** (normalise → blocking → learned pruning → LightGBM matching) on a **small fake dataset**. The real Amazon ML Challenge 2026 data isn't public.

Click **Runtime → Run all**. No GPU and no uploads are needed.

## 1. Get the code and install the libraries

In [ ]:
REPO = "https://github.com/Sashank2006/Amazon-ML-Entity-Resolution"
!git clone -q {REPO} er && echo "cloned"
%cd er
!pip install -q rapidfuzz lightgbm indic-transliteration

## 2. Create a fake dataset with realistic noise
It includes typos, `0/o` swaps, `Pvt Ltd` ↔ `Private Limited`, reordered words, `name.com`-style names, dropped house numbers, empty addresses, look-alike decoys, and a third country (France) that appears **only** in the test set.

In [ ]:
!python demo/make_demo_data.py --out demo_data --n-train 6000 --n-test 3000
import pandas as pd
pd.read_csv("demo_data/train/train_source2.tsv", sep="\t").head(8)

## 3. Run the whole pipeline (about 20 seconds)

In [ ]:
!python src/pipeline.py all --data demo_data --work demo_cache --model-dir demo_model --out demo_output

## 4. Score it and look at some matches

In [ ]:
!python demo/show_results.py --data demo_data --out demo_output

## What just happened?
| Stage | What it does |
|---|---|
| `prep` | Cleans the text: accents, legal suffixes, Rd→rd, Indian scripts → Latin letters |
| `block` | Rare shared "keys" (house number + street word, name prefixes, sound-alike codes) produce ~30 candidates per business instead of comparing every pair |
| `train` | Stage 1: a small model prunes the candidates to about 3–5. Stage 2: the matching model on ~50 similarity features |
| `predict` | Scores the test pairs, gives each record to at most one business, and writes `matching_results.tsv` + `candidate_pairs.tsv` |

On the real challenge (1.7M businesses vs 10M records, free Colab CPU) the same code reached a **public leaderboard score of 0.935 (macro F0.5)**. See `docs/METHODOLOGY.md`.